# Notebook 5 — Aggregation Features
Customer-level aggregations built from the transaction log (`telecom_transactions.csv`)
and joined back onto `telecom_customers.csv`. This is exactly the pattern used in
real fraud, churn, and CLV (customer lifetime value) models at any subscription
company.

In [ ]:
import pandas as pd
import numpy as np

customers = pd.read_csv("./telecom_customers.csv", parse_dates=["signup_date"])
transactions = pd.read_csv("./telecom_transactions.csv", parse_dates=["transaction_date"])
transactions.head()

## Why Aggregation Features Matter

Raw transaction-level data can't be fed directly into a customer-level churn model —
there's a variable number of transactions per customer. Aggregation collapses the
transaction log into **one row per customer**, summarizing behavior with statistics
that *are* usable as model input.

## 1. Customer-Level Aggregation (Count, Sum, Mean, Median, Min, Max, Std)

In [ ]:
agg = transactions.groupby("customer_id")["amount"].agg(
    total_spent="sum",
    avg_transaction_amount="mean",
    median_transaction_amount="median",
    min_transaction_amount="min",
    max_transaction_amount="max",
    std_transaction_amount="std",
    num_transactions="count",
).reset_index()

agg["std_transaction_amount"] = agg["std_transaction_amount"].fillna(0)  # single-tx customers
agg.head()

**Business meaning of each statistic:**
- `total_spent` — overall value of the customer (raw magnitude of engagement)
- `avg_transaction_amount` — typical spend per interaction
- `std_transaction_amount` — spend *volatility*; a customer with wildly inconsistent
  transaction sizes behaves differently from one with a stable pattern
- `num_transactions` — engagement frequency, a strong loyalty/stickiness proxy
- `max_transaction_amount` — captures one-off large purchases (e.g. device upgrades)
  that a mean would dilute

## 2. Product / Transaction-Type Level Aggregation

**When to use:** when *what* a customer buys matters as much as *how much* — e.g.
distinguishing routine bill payments from proactive add-on purchases (a sign of
engagement, not just obligation).

In [ ]:
type_pivot = transactions.pivot_table(
    index="customer_id", columns="transaction_type", values="amount",
    aggfunc="count", fill_value=0
)
type_pivot.columns = [f"count_{c.lower().replace(' ','_')}" for c in type_pivot.columns]
type_pivot = type_pivot.reset_index()
type_pivot.head()

In [ ]:
customers = customers.merge(agg, on="customer_id", how="left").merge(type_pivot, on="customer_id", how="left")

agg_cols = list(agg.columns[1:]) + list(type_pivot.columns[1:])
customers[agg_cols] = customers[agg_cols].fillna(0)

customers[["customer_id","total_spent","num_transactions","count_add-on_purchase" if "count_add-on_purchase" in customers.columns else agg_cols[-1]]].head()

## 3. Derived Ratio-on-Aggregation Features

Combining an aggregation with another customer attribute produces some of the most
powerful engineered features in the whole pipeline — e.g. **spend rate relative to
tenure**.

In [ ]:
customers["avg_spend_per_month_active"] = customers["total_spent"] / customers["tenure_months"].replace(0, 1)
customers["addon_purchase_ratio"] = (
    customers.get("count_add-on_purchase", 0) / customers["num_transactions"].replace(0, 1)
)

customers[["customer_id","avg_spend_per_month_active","addon_purchase_ratio"]].sort_values(
    "avg_spend_per_month_active", ascending=False
).head()

## Summary — Feature Justification

| Feature | Source | Logic | Business Meaning | Leakage Risk | Decision |
|---|---|---|---|---|---|
| `total_spent` | transactions.amount (sum) | group-by sum | overall customer value | Must only use transactions **before** prediction date in production | **Retain** |
| `std_transaction_amount` | transactions.amount (std) | group-by std | spend volatility | Same as above | **Retain** |
| `num_transactions` | transactions (count) | group-by count | engagement frequency | Same as above | **Retain** |
| `avg_spend_per_month_active` | total_spent, tenure_months | ratio | normalized spend rate | Inherits leakage risk from total_spent | **Retain** |

> **Critical leakage warning (previewing Notebook 12):** these aggregations must only
> include transactions that occurred **before** the point in time we're predicting
> churn for. Aggregating the *entire* transaction history (including transactions that
> happen after a customer has already churned) is a textbook example of temporal
> leakage. In this notebook we aggregate the full log for teaching simplicity — the
> leakage-safe version is built explicitly in Notebook 12 and 13.